[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamaly/dpgs-analysis-2026/blob/main/03_ai_features.ipynb)


In [1]:
# ======================================================================================================
#  SETUP (inserted by the build script). In Google Colab this downloads the repository so that the
#  `data/` folder sits next to the notebook; on your own computer it does nothing at all.
# ======================================================================================================
import os, sys

REPO_SLUG = "gamaly/dpgs-analysis-2026"     # <github user>/<repository>; set in scripts/99b_build_public_repo.py
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules      # Colab imports this module in every session; a local Jupyter does not

if IN_COLAB and not os.path.isdir("data"):
    # Colab starts with an empty working directory, so the data folder is missing. Fetch the whole repository
    # (a few tens of MB): first try a shallow `git clone`, and if that fails download the branch zip instead.
    import subprocess, urllib.request, zipfile
    repo_dir = REPO_SLUG.split("/")[-1]
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://github.com/{REPO_SLUG}.git", repo_dir], check=True)
    except Exception as err:
        print("git clone failed (", err, ") -> downloading the zip instead")
        url = f"https://github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip"
        urllib.request.urlretrieve(url, "repo.zip")
        with zipfile.ZipFile("repo.zip") as z:
            z.extractall(".")                                   # unpacks to <repository>-<branch>/
        repo_dir = f"{repo_dir}-{BRANCH}"
    os.chdir(repo_dir)                                         # from here on, "data/" and "extras/" resolve as they do locally
    print("Colab: working in", os.getcwd())
    # Colab already has pandas, numpy and matplotlib. Install only what is missing (nothing, normally).
    import importlib.util
    missing = [p for p in ("pandas", "numpy", "matplotlib") if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif not IN_COLAB:
    # Local use: run Jupyter from the repository folder (the one containing this notebook and data/).
    assert os.path.isdir("data"), "Start Jupyter in the repository folder: 'data/' must sit next to this notebook."


# 03. AI features inside DPG products

**What this notebook does.** It answers: *are DPG teams building AI into their products, and does it reach users?*
It counts merged pull requests (PRs) that add AI features, checks how many of them reached a release, and counts
DPGs that mention AI features in their own release notes.

**How "AI feature PR" is defined (METHODOLOGY D36, D48; script `03f_ai_feature_prs.py`).**
A merged, human-authored PR is an *AI-feature PR* if its **title or cleaned body** matches a fixed list of terms
(chatbot, LLM, GPT, OpenAI, RAG, embeddings, speech-to-text, machine learning, NLP, computer vision, ...).
Before matching, body lines about *using* AI to write the code (co-author trailers, "generated with", tool names,
checklists, AI contribution policies, `AGENTS.md`) are removed, so AI-assisted *coding* is not mistaken for an
AI *feature*. Hand-checked precision: about 85% for title matches and 83% for body-only matches (D48).
This is **a descriptive count, not a test**, and keyword matching misses undisclosed or unusually phrased work.

**Shipped** (script `03f`/`03g`): an AI-feature PR counts as shipped if a release or tag was published after its
merge; `days_to_release` is the gap to the first such release.

**Release notes** (`03g_release_ai.py`): GitHub release notes that name an AI feature, one row per release.

**Privacy.** The released PR files have no free text: the `context` column (quoted PR/release text) is removed and
PR titles are blanked except for the few the post quotes or paraphrases.

**Inputs:** `ai_feature_prs.csv`, `ai_feature_prs_shipped.csv`, `ai_feature_prs_yearly.csv`, `release_notes_ai.csv`,
`dpg_confidence.csv`.

In [2]:
import sys
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
DATA = Path("data")
sys.path.insert(0, "extras")
import ledger_check as lc

## 1. Load the data
`confidence` in these files is the *repo* tier (high / medium). The main results use `high` (D14). A second filter,
"DPG is in the 166", gives the same answer here because every high-confidence repo of a medium DPG is excluded by the
matching in the original script; both are printed.

In [3]:
ai = pd.read_csv(DATA / "ai_feature_prs.csv")
sh = pd.read_csv(DATA / "ai_feature_prs_shipped.csv")
yr = pd.read_csv(DATA / "ai_feature_prs_yearly.csv")
rn = pd.read_csv(DATA / "release_notes_ai.csv")
conf = pd.read_csv(DATA / "dpg_confidence.csv")
high_dpgs = set(conf.loc[conf.dpg_confidence == "high", "dpgid"])

ai_h = ai[ai.confidence == "high"].copy()
ai_h["year"] = ai_h.merged_at.str[:4].astype(int)
print("AI-feature PR rows (all tiers):", len(ai), "| repo tier high:", len(ai_h))
print("...also restricted to the 166 DPGs:", ai_h.dpgid.isin(high_dpgs).sum())

AI-feature PR rows (all tiers): 1123 | repo tier high: 1120
...also restricted to the 166 DPGs: 1119


### Claim N020: DPGs with at least one merged AI-feature PR (of the 166)

In [4]:
n_dpgs = ai_h.dpgid.nunique()
print(n_dpgs, "DPGs (of", len(high_dpgs), ")")
lc.claim("N020", "DPGs with a merged AI-feature PR (high tier)", [n_dpgs])

68 DPGs (of 166 )


### Claim N021: of those, DPGs whose first such PR was in 2024 or later
"First time in 2024 or later" = the earliest merge year of any AI-feature PR of the DPG is 2024 or more.

In [5]:
first_year = ai_h.groupby("dpgid").year.min()
n_new = int((first_year >= 2024).sum())
print(first_year.value_counts().sort_index().to_string())
lc.claim("N021", "DPGs whose first AI-feature PR is in 2024 or later", [n_new])

year
2020     8
2021     9
2022     1
2023     7
2024    11
2025    19
2026    13


### Claim N022: DPGs doing this work per year, 2020-2026
From the yearly file (`dpgs_with_ai_feature_pr`). Before 2024 never more than 16 in a year; then 18, 37, and 51 in
the first ten months of 2026 (to 29 October).

In [6]:
yh = yr[yr.confidence == "high"].set_index("year")
print(yh[["merged_prs", "ai_feature_prs", "dpgs", "dpgs_with_ai_feature_pr"]].to_string())
series = {y: int(yh.dpgs_with_ai_feature_pr[y]) for y in range(2020, 2027)}
print(series)
lc.claim("N022", "DPGs with an AI-feature PR per year 2020..2026 (+ the years 2020 and 2023 named in the ledger text)",
         [series[y] for y in range(2020, 2027)] + [2020, 2023])

      merged_prs  ai_feature_prs  dpgs  dpgs_with_ai_feature_pr
year                                                           
2020       30794              47    79                        8
2021       31361              50    93                       13
2022       34249              24   114                        8
2023       42581              39   123                       16
2024       42862             101   130                       18
2025       43860             281   122                       37
2026       50631             578   117                       51
{2020: 8, 2021: 13, 2022: 8, 2023: 16, 2024: 18, 2025: 37, 2026: 51}


### Claim N023: AI-feature PRs, 2023 vs 2026

In [7]:
lc.claim("N023", "AI-feature PRs in 2023 and 2026", [yh.ai_feature_prs[2023], yh.ai_feature_prs[2026]])
print(yh.ai_feature_prs[[2023, 2026]].to_dict())

{2023: 39, 2026: 578}


### Claim N024: still about one merged change in ninety
2026 AI-feature PRs divided by all merged human PRs of the high-confidence repos that year.

In [8]:
ratio = yh.ai_feature_prs[2026] / yh.merged_prs[2026]
print(f"{yh.ai_feature_prs[2026]} / {yh.merged_prs[2026]} = {ratio:.4f}  (1 in {1/ratio:.0f})")
lc.claim("N024", "1 in N and percent (2026)", [1, 1 / ratio, ratio * 100])

578 / 50631 = 0.0114  (1 in 88)


## 2. Did the AI features reach users?
`ai_feature_prs_shipped.csv` adds `shipped` (a later release/tag exists), `first_release_date`, `days_to_release`.

In [9]:
sh_h = sh[sh.confidence == "high"].copy()
sh_h["shipped"] = sh_h.shipped.astype(str) == "True"
print("PRs:", len(sh_h), "| shipped:", int(sh_h.shipped.sum()))

PRs: 1120 | shipped: 962


### Claim N025: share of AI-feature PRs that reached a release

In [10]:
pct_shipped = sh_h.shipped.mean() * 100
print(round(pct_shipped, 1))
lc.claim("N025", "percent shipped", [pct_shipped])

85.9


### Claim N026: median days from merge to the first release

In [11]:
med_days = sh_h.loc[sh_h.shipped, "days_to_release"].median()
print(med_days)
lc.claim("N026", "median days to release (shipped PRs)", [med_days])

6.0


### Claim N027 (NOT IN THE POST: dropped in v5, kept for reference): DPGs that put at least one into a release

In [12]:
n_ship = sh_h[sh_h.shipped].dpgid.nunique()
n_all = sh_h.dpgid.nunique()
print(n_ship, "of", n_all)
lc.claim("N027", "DPGs with a shipped AI-feature PR; DPGs with any", [n_ship, n_all])

62 of 68


## 3. Release notes
A DPG "names an AI feature in its own release notes" if at least one of its releases matches. The post says
thirty DPGs, 28 of them since 2024. Two definitions of "since 2024" are possible: any AI-feature release dated 2024
or later (28), or the *first* such release in 2024 or later (23). The post uses the first (any release since 2024).

### Claim N028 (NOT IN THE POST: dropped in v5, kept for reference): DPGs naming an AI feature in release notes

In [13]:
rn_h = rn[rn.dpgid.isin(high_dpgs)].copy()
rn_h["year"] = rn_h.published_at.str[:4].astype(int)
n_rn = rn_h.dpgid.nunique()
print(n_rn, "DPGs;", len(rn_h), "release rows")
lc.claim("N028", "DPGs with AI feature in release notes", [n_rn])

30 DPGs; 165 release rows


### Claim N029 (NOT IN THE POST: dropped in v5, kept for reference): of those, DPGs with such a release dated 2024 or later

In [14]:
n_rn24 = rn_h[rn_h.year >= 2024].dpgid.nunique()
n_rn_first24 = int((rn_h.groupby("dpgid").year.min() >= 2024).sum())
print("any release in 2024+:", n_rn24, "| first AI release in 2024+:", n_rn_first24)
lc.claim("N029", "DPGs with an AI-feature release in 2024 or later", [n_rn24])

any release in 2024+: 28 | first AI release in 2024+: 23


## 4. Concentration and examples

### Claim N030: KoboToolbox alone accounts for 196 of the 1,120 PRs

In [15]:
kobo = int((ai_h.dpg == "KoboToolbox").sum())
print(kobo, "of", len(ai_h))
lc.claim("N030", "KoboToolbox AI-feature PRs; all AI-feature PRs (high tier)", [kobo, len(ai_h)])

196 of 1120


### Claim N031: the CommCare PR that moved AI translations off GPT-4.1
PR titles are kept in the release only for the PRs the post quotes.

In [16]:
t = "Move AI translations off gpt-4.1 before its shutdown"
hit = ai_h[(ai_h.dpg == "CommCare") & (ai_h.title == t) & (ai_h.year == 2026)]
print(hit[["dpg", "full_name", "merged_at", "title"]].to_string(index=False))
lc.claim("N031", "number of matching PRs (CommCare, 2026, exact title)", [len(hit)])

     dpg          full_name            merged_at                                                title
CommCare dimagi/commcare-hq 2026-10-23T06:18:24Z Move AI translations off gpt-4.1 before its shutdown


### Claim N032: named examples
The post describes one example per DPG (paraphrasing the PR title). The check is that a title of that kind exists
for each named DPG. Only these titles remain in the release file (all other titles are blanked).

In [17]:
examples = {   # DPG -> a distinctive piece of the PR title the post paraphrases
    "KoboToolbox": "automatic transcription and translation",
    "Glific": "feature/openai",
    "CommCare": "create translations in hq with llms",
    "La Suite Meet": "stt",
    "Chamilo": "ai-generated scorm",
}
titles = ai_h.assign(t=ai_h.title.fillna("").str.lower())
checks = {k: bool(titles[(titles.dpg == k) & titles.t.str.contains(v, regex=False)].shape[0]) for k, v in examples.items()}
print(checks)
lc.claim("N032", "named example DPGs have a matching AI-feature PR", [], ok=all(checks.values()))

{'KoboToolbox': True, 'Glific': True, 'CommCare': True, 'La Suite Meet': True, 'Chamilo': True}


## 5. Summary: reproduced value next to the ledger value

In [18]:
_ = lc.show("03_ai_features")

claim                         ledger value                                  reproduced here                                            detail           grade
 N020                                   68                                               68                                                             exact
 N021                                   43                                               43                                                             exact
 N022 8; 13; 8; 16 (2020-2023); 18; 37; 51             8, 13, 8, 16, 18, 37, 51, 2020, 2023                                                             exact
 N023                              39; 578                                          39, 578                                                             exact
 N024                      1 in 88 (1.14%)                              1, 87.5969, 1.14159                                                          rounding
 N025                                 85.9          